In [257]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
from ast import literal_eval
from collections import Counter
import math

In [220]:
username = os.path.expanduser('~')
username

'/root'

In [ ]:
benchmarks = [
    # High-sharing Apps
    'jacobi2d',
    'stencil2d',
    'gemm',
    'lu',
    # 'spmv',
    'mm3',
    'mm2',
    'matrixmultiplication',
    'syr2k',
    'gups',    
    # 'syrk',
    'mis', # Deadlock
    'matrixtranspose', # Error
    # 'gaussian',
    # 'b+tree',

    # # Low-sharing Apps
    'fdtd2d',
    'fastwalshtransform',
    'jacobi1d',
    'relu',
    'pagerank',
    # 'shoc-reduction',
    'bfs',
    # # 'nn',
    # # 'hotspot3d',
    # # 'srad',
    # 'color',
    # 'convolution2d',
    'sssp',
]

In [ ]:
num_chiplets = 36
ns = 0.000000001

In [ ]:
base_dir=f'{username}/mgvm/results/memory-traces'
date = ""
print(base_dir)
print(result_dir)

In [ ]:
def parse_metrics(s):
    try:
        return literal_eval(s)
    except Exception as e:
        print(f"Parsing error: {s} -> {e}")
        return {}

,time,addr,count,chiplets
0,0.0,2164035840,2,1
1,2.0,2164035840,2,1
2,4.0,2164035840,2,1
3,6.0,2164035840,2,1
4,7.0,2164035840,1,1
...,...,...,...,...
688111,40729.0,2147561216,1,1
688112,40748.0,2147561216,1,1
688113,40750.0,2147561280,1,1
688114,40752.0,2147561344,1,1


In [ ]:
for benchmark in benchmarks:
    result_dir=f'{username}/mgvm/results/memory-traces/{date}/data'

    if not os.path.isdir(f'{result_dir}'):
        os.makedirs(f'{result_dir}', exist_ok=True, mode=0o777)
    

    trace_file = f"{base_dir}/{date}/{benchmark}_mesh_6x6_memory_trace.csv"

    df = pd.read_csv(
        trace_file,
        skipinitialspace=True,
        converters={"metrics": parse_metrics}  # 읽으면서 바로 딕셔너리로 변환
    )

    expanded = []
    for idx, row in df.iterrows():
        time = row['time']
        for addr, (count, chiplets) in row['metrics'].items():
            expanded.append({
                'time': time,
                'addr': addr,
                'count': count,
                'chiplets': chiplets
            })

    df = pd.DataFrame(expanded)
    df['time'] = df['time']-df['time'].min()
    df['time'] /= ns
    total_cycles = df['time'].max()-df['time'].min()

    term_cycles = 1000
    division = math.ceil(total_cycles / term_cycles)

    for p in range(0, division):
        period_df = df[(df['time']>(p*term_cycles)) & (df['time']<=((p+1)*term_cycles))]
        shared_df = period_df.groupby('addr', as_index=False)['count'].sum()
        shared_df = shared_df[shared_df['count']>1]

        total_period_count = period_df['count'].sum()
        total_shared_count = shared_df['count'].sum()

        period_start = p*term_cycles
        period_end = (p+1)*term_cycles
        print(f"{period_start:0.5f}~{period_end:0.5f}: {total_shared_count/total_period_count}")


NameError: name 'benchmarks' is not defined

In [ ]:
trace_file = f"/root/mgvm/mgpusim/samples/pagerank/pagerank_mesh_6x6_memory_trace.csv"

df = pd.read_csv(
    trace_file,
    skipinitialspace=True,
    converters={"metrics": parse_metrics}  # 읽으면서 바로 딕셔너리로 변환
)

expanded = []
for idx, row in df.iterrows():
    time = row['time']
    for addr, (count, chiplets) in row['metrics'].items():
        expanded.append({
            'time': time,
            'addr': addr,
            'count': count,
            'chiplets': chiplets
        })

df = pd.DataFrame(expanded)
df['time'] = df['time']-df['time'].min()
df['time'] /= ns
total_cycles = df['time'].max()-df['time'].min()

term_cycles = 1000
division = math.ceil(total_cycles / term_cycles)

results = []

for p in range(0, division):
    period_df = df[(df['time']>(p*term_cycles)) & (df['time']<=((p+1)*term_cycles))]
    shared_df = period_df.groupby('addr', as_index=False)['count'].sum()
    shared_df = shared_df[shared_df['count']>1]

    total_addr = len(period_df)
    total_shared_addr = len(shared_df)

    total_count = period_df['count'].sum()
    total_shared_count = shared_df['count'].sum()

    period_start = p*term_cycles
    period_end = (p+1)*term_cycles
    # print(f"{period_start:0.5f}~{period_end:0.5f}: {total_shared_count/total_period_count}")

        
    results.append({
        "start": period_start,
        "end": period_end if period_end < df['time'].max() else df['time'].max(),
        "shared_addr_ratio": total_shared_addr / total_addr if total_addr > 0 else 0,
        "shared_count_ratio": total_shared_count / total_count if total_count > 0 else 0 
    })

results_df = pd.DataFrame(results)
results_df.to_csv()

In [279]:
results_df

,start,end,shared_addr_ratio,shared_count_ratio
0,0,1000.0,0.046053,1.000000
1,1000,2000.0,0.201976,1.000000
2,2000,3000.0,0.149882,0.993754
3,3000,4000.0,0.274419,0.993127
4,4000,5000.0,0.310106,0.997243
5,5000,6000.0,0.369298,0.996456
6,6000,7000.0,0.367961,0.992784
7,7000,8000.0,0.397212,0.993683
8,8000,9000.0,0.392771,0.994855
9,9000,10000.0,0.390384,0.994754
